In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================
# CẤU HÌNH
# ============================================================

verifier_id = "Qwen/Qwen2.5-7B-Instruct"
drafter_id = "Efficient-Large-Model/Fast_dLLM_v2_1.5B"

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

# ============================================================
# LOAD QWEN 2.5 7B
# ============================================================

print("Loading tokenizer Qwen...")

tokenizer_verifier = AutoTokenizer.from_pretrained(
    verifier_id,
    trust_remote_code=True
)

print("Loading model Qwen 2.5 7B...")

model_verifier = AutoModelForCausalLM.from_pretrained(
    verifier_id,
    device_map="auto",
    torch_dtype=dtype,
    trust_remote_code=True
)

model_verifier.eval()

# ============================================================
# LOAD FAST-dLLM 1.5B
# ============================================================

print("\nLoading tokenizer Fast-dLLM...")

tokenizer_drafter = AutoTokenizer.from_pretrained(
    drafter_id,
    trust_remote_code=True
)

print("Loading model Fast-dLLM 1.5B...")

model_drafter = AutoModelForCausalLM.from_pretrained(
    drafter_id,
    device_map="auto",
    torch_dtype=dtype,
    trust_remote_code=True
)

model_drafter.eval()

if torch.cuda.is_available():
    print(
        f"\nVRAM đã cấp phát: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )

# ============================================================
# PROMPT DÙNG CHUNG
# ============================================================

problem = (
    "Natalia sold clips to 48 of her friends in April, "
    "and then she sold half as many clips in May. "
    "How many clips did Natalia sell altogether in April and May?"
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a careful mathematics tutor. "
            "Solve the problem accurately and explain the reasoning clearly."
        )
    },
    {
        "role": "user",
        "content": (
            problem
            + "\n\nPlease solve the problem step by step. "
              "Show the relevant calculations, and finish with a clearly "
              "marked final answer in the form: Final answer: <answer>."
        )
    }
]


# ============================================================
# HÀM SINH CHO QWEN
# ============================================================

def generate_qwen(messages, max_new_tokens=256):
    inputs = tokenizer_verifier.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Chuyển input tới thiết bị chứa embedding layer
    input_device = model_verifier.get_input_embeddings().weight.device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model_verifier.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_verifier.eos_token_id
        )

    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]

    return tokenizer_verifier.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]


# ============================================================
# HÀM SINH CHO FAST-dLLM
# ============================================================

def generate_fast_dllm(
    messages,
    max_new_tokens=256,
    small_block_size=8,
    threshold=0.9
):
    # Model card Fast-dLLM dùng:
    # apply_chat_template(..., tokenize=False)
    # rồi tokenize chuỗi đã format.
    prompt_text = tokenizer_drafter.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_drafter(
        prompt_text,
        return_tensors="pt"
    )

    input_device = model_drafter.get_input_embeddings().weight.device
    input_ids = inputs["input_ids"].to(input_device)

    with torch.inference_mode():
        output_ids = model_drafter.generate(
            input_ids,
            tokenizer=tokenizer_drafter,
            max_new_tokens=max_new_tokens,
            small_block_size=small_block_size,
            threshold=threshold
        )

    new_tokens = output_ids[:, input_ids.shape[1]:]

    return tokenizer_drafter.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]


# ============================================================
# TEST
# ============================================================

print("\n" + "=" * 70)
print("QWEN 2.5 7B")
print("=" * 70)
print(generate_qwen(messages))

print("\n" + "=" * 70)
print("FAST-dLLM V2 1.5B")
print("=" * 70)
print(generate_fast_dllm(messages))